<a href="https://colab.research.google.com/github/varsshinii35cyber/chatbot-codealpha/blob/main/chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Step 1: Install Required Libraries**

In [7]:
!pip install -q langchain langchain-community langchain-google-genai
!pip install -q faiss-cpu sentence-transformers pypdf nltk spacy
!python -m spacy download en_core_web_sm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 84.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.8/72.8 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 57.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.7/561.7 kB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 90.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 30.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 81.5 MB/s eta 0:00:00
✔ Download and in

In [10]:
!pip install langchain langchain-community

In [13]:
import os
import re
import pickle
import getpass

import numpy as np
import pandas as pd

import nltk
import spacy

from google.colab import files

from pypdf import PdfReader

from sentence_transformers import SentenceTransformer

from langchain_text_splitters import RecursiveCharacterTextSplitter

import faiss

nltk.download("stopwords")

from nltk.corpus import stopwords

nlp = spacy.load("en_core_web_sm")

stop_words = set(stopwords.words("english"))

print("Setup completed")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Setup completed


In [15]:
uploaded = files.upload()

pdf_files=list(uploaded.keys())

print(pdf_files)

Saving python to python
['python']


In [28]:
def extract_pdf(file):

    text=""

    reader=PdfReader(file)

    for page in reader.pages:

        page_text=page.extract_text()

        if page_text:
            text += page_text + "\n"

    return text



raw_text=""


for pdf in pdf_files:

    print("Processing:", pdf)

    raw_text += extract_pdf(pdf)


print("Total characters:", len(raw_text))

Processing: python
Total characters: 347678


In [29]:
def clean_text(text):

    text=text.lower()

    text=re.sub(
        r"\s+",
        " ",
        text
    )

    return text



cleaned_text = clean_text(raw_text)



def preprocess(text):

    doc = nlp(text)

    tokens=[]

    for token in doc:

        if not token.is_stop and not token.is_punct:

            tokens.append(token.lemma_)


    return " ".join(tokens)



processed_text = preprocess(cleaned_text)


print(processed_text[:500])

introductiontopythonfor computationalscienceand engineering hansfangohr apr14,2024 content 1 introduction 3 1.1 computational modelling 3 1.2 python scientific computing 5 1.3 python version 7 1.4 document 8 1.5 feedback 9 2 powerful calculator 11 2.1 python prompt read eval print loop repl 11 2.2 calculator 11 2.3 integer division 13 2.4 mathematical function 14 2.5 variable 16 2.6 impossible equation 18 3 datum type datum structure 21 3.1 type 21 3.2 number 21 3.3 sequence 24 3.4 pass argument


In [30]:
splitter = RecursiveCharacterTextSplitter(

    chunk_size=500,

    chunk_overlap=100
)


documents = splitter.split_text(
    processed_text
)


print(
    "FAQ chunks:",
    len(documents)
)

FAQ chunks: 599


In [31]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)


embeddings = embedding_model.encode(
    documents,
    show_progress_bar=True
)


embeddings = np.array(
    embeddings
).astype("float32")


print(embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/19 [00:00<?, ?it/s]

(599, 384)


In [32]:
dimension = embeddings.shape[1]


faiss_index = faiss.IndexFlatL2(
    dimension
)


faiss_index.add(
    embeddings
)


print("FAISS ready")

FAISS ready


In [33]:
faiss.write_index(
    faiss_index,
    "faq.index"
)


with open(
    "faq_documents.pkl",
    "wb"
) as f:

    pickle.dump(
        documents,
        f
    )


print("Saved")

Saved


In [34]:
def chatbot(question):


    question_embedding = embedding_model.encode(
        [question]
    )


    question_embedding = np.array(
        question_embedding
    ).astype("float32")


    distance, result = faiss_index.search(
        question_embedding,
        1
    )


    answer = documents[result[0][0]]


    confidence = 1/(1+distance[0][0])


    return answer, confidence

In [35]:
print("🤖 FAQ Chatbot")
print("Type exit to stop")


while True:

    user=input("\nYou: ")


    if user.lower()=="exit":

        print("Bot: Goodbye!")

        break


    answer, confidence = chatbot(user)


    print("\nBot:")
    print(answer)

    print(
        "Confidence:",
        round(confidence*100,2),
        "%"
    )

🤖 FAQ Chatbot
Type exit to stop

You: what is python

Bot:
section jacek generowicz introduce python millennium kindly share countless idea excellent python course epsrc gr t09156/01 ep g03690x/1 european union opendreamkit horizon 2020 european research infrastructure project 676541 support student reader provide feedback point typo error etc thomas kluyver help translate python 2 latex base document python 3 jupyter notebook provide machinery create html pdf version automatically bookbook package 258 chapter19 wheretogofromhere
Confidence: 51.09 %

You: python prompt?

Bot:
.py example hello.py enter individual command python prompt immediately evaluate carry python interpreter useful programmer learner understand use certain command put command long python program python role describe read command evaluate print evaluated value repeat loop cycle origin repl abbreviation python come basic terminal prompt example > > > mark input > > > 2 + 2 4 powerful repl interface jupyter notebook 